# RetailMind AI — Day 4: Feature Engineering, PCA & Ensembles
**Objective**: Add tenure buckets, charge ratios, PCA dimension reduction, train XGBoost boosting ensemble, and freeze the production churn pipeline.

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, recall_score, accuracy_score

In [ ]:
# Load production model persisted in Day 4
prod_pipeline = joblib.load('../models/churn_production_v4.joblib')
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df = df.dropna(subset=['TotalCharges']).copy()
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
preds = prod_pipeline.predict(X_test)
probas = prod_pipeline.predict_proba(X_test)[:, 1]

print(f"Final Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Final Recall:   {recall_score(y_test, preds):.4f}")
print(f"Final ROC-AUC:  {roc_auc_score(y_test, probas):.4f}")
print("\nClassification Report:\n", classification_report(y_test, preds))